In [1]:
# Cell 1 — Imports and Load Processed Data
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Load raw data
patients = pd.read_csv('../data/raw/csv/patients.csv')
encounters = pd.read_csv('../data/raw/csv/encounters.csv')
conditions = pd.read_csv('../data/raw/csv/conditions.csv')
medications = pd.read_csv('../data/raw/csv/medications.csv')
procedures = pd.read_csv('../data/raw/csv/procedures.csv')

# Convert dates
patients['BIRTHDATE'] = pd.to_datetime(patients['BIRTHDATE'])
encounters['START'] = pd.to_datetime(encounters['START'])
encounters['STOP'] = pd.to_datetime(encounters['STOP'])
conditions['START'] = pd.to_datetime(conditions['START'])

print("Data loaded successfully!")
print(f"Patients: {patients.shape}")
print(f"Encounters: {encounters.shape}")

Data loaded successfully!
Patients: (1171, 25)
Encounters: (53346, 15)


In [2]:
# Cell 2 — Build Base Feature Table
# Filter inpatient encounters only
inpatient = encounters[encounters['ENCOUNTERCLASS'] == 'inpatient'].copy()
inpatient = inpatient.sort_values(['PATIENT', 'START']).reset_index(drop=True)

# Target variable — 30 day readmission
inpatient['NEXT_ADMISSION'] = inpatient.groupby('PATIENT')['START'].shift(-1)
inpatient['DAYS_TO_READMISSION'] = (inpatient['NEXT_ADMISSION'] - inpatient['STOP']).dt.days
inpatient['READMITTED_30'] = (inpatient['DAYS_TO_READMISSION'] <= 30).astype(int)

# Length of stay in hours (capped at 99th percentile)
inpatient['LENGTH_OF_STAY'] = (inpatient['STOP'] - inpatient['START']).dt.total_seconds() / 3600
los_cap = inpatient['LENGTH_OF_STAY'].quantile(0.99)
inpatient['LENGTH_OF_STAY'] = inpatient['LENGTH_OF_STAY'].clip(upper=los_cap)

# Prior admissions count
inpatient['PRIOR_ADMISSIONS'] = inpatient.groupby('PATIENT').cumcount()

print(f"Base table shape: {inpatient.shape}")
print(f"Readmission rate: {inpatient['READMITTED_30'].mean():.2%}")
print(f"LOS cap at 99th percentile: {los_cap:.1f} hours")

Base table shape: (1838, 20)
Readmission rate: 41.84%
LOS cap at 99th percentile: 96.7 hours


In [5]:
# Cell 3 — Demographics Features (fixed)
# Strip timezone from START and STOP
inpatient['START'] = inpatient['START'].dt.tz_localize(None)
inpatient['STOP'] = inpatient['STOP'].dt.tz_localize(None)

# Prepare patients dataframe
patients['BIRTHDATE'] = pd.to_datetime(patients['BIRTHDATE'])
patients_subset = patients[['Id', 'BIRTHDATE', 'GENDER', 'RACE', 
                              'ETHNICITY', 'MARITAL', 
                              'HEALTHCARE_EXPENSES', 'HEALTHCARE_COVERAGE']].copy()

# Merge
inpatient = inpatient.merge(patients_subset, left_on='PATIENT', 
                             right_on='Id', how='left', suffixes=('', '_patient'))

# Age at time of admission
inpatient['AGE'] = (inpatient['START'] - inpatient['BIRTHDATE']).dt.days // 365

# Encode categorical variables
inpatient['GENDER_ENCODED'] = (inpatient['GENDER'] == 'M').astype(int)
inpatient['RACE_ENCODED'] = inpatient['RACE'].map({
    'white': 0, 'black': 1, 'asian': 2, 'native': 3, 'other': 4
}).fillna(0)
inpatient['MARITAL_ENCODED'] = (inpatient['MARITAL'] == 'M').astype(int)

print("Demographics features added!")
print(inpatient[['AGE', 'GENDER_ENCODED', 'RACE_ENCODED', 
                  'MARITAL_ENCODED', 'HEALTHCARE_EXPENSES']].describe())

Demographics features added!
               AGE  GENDER_ENCODED  RACE_ENCODED  MARITAL_ENCODED  \
count  1838.000000     1838.000000   1838.000000       1838.00000   
mean     49.555495        0.437976      0.375408          0.70185   
std      21.963669        0.496273      0.815263          0.45757   
min       1.000000        0.000000      0.000000          0.00000   
25%      31.000000        0.000000      0.000000          0.00000   
50%      53.000000        0.000000      0.000000          1.00000   
75%      65.000000        1.000000      0.000000          1.00000   
max     110.000000        1.000000      3.000000          1.00000   

       HEALTHCARE_EXPENSES  
count         1.838000e+03  
mean          9.223696e+05  
std           6.014573e+05  
min           7.324040e+03  
25%           2.689206e+05  
50%           1.075828e+06  
75%           1.469461e+06  
max           1.863215e+06  


In [6]:
# Cell 4 — Clinical Complexity Features
# Total conditions per patient
condition_counts = conditions.groupby('PATIENT').size().reset_index(name='TOTAL_CONDITIONS')
inpatient = inpatient.merge(condition_counts, on='PATIENT', how='left')
inpatient['TOTAL_CONDITIONS'] = inpatient['TOTAL_CONDITIONS'].fillna(0)

# Total medications per patient
med_counts = medications.groupby('PATIENT').size().reset_index(name='TOTAL_MEDICATIONS')
inpatient = inpatient.merge(med_counts, on='PATIENT', how='left')
inpatient['TOTAL_MEDICATIONS'] = inpatient['TOTAL_MEDICATIONS'].fillna(0)

# Total procedures per patient
proc_counts = procedures.groupby('PATIENT').size().reset_index(name='TOTAL_PROCEDURES')
inpatient = inpatient.merge(proc_counts, on='PATIENT', how='left')
inpatient['TOTAL_PROCEDURES'] = inpatient['TOTAL_PROCEDURES'].fillna(0)

# Prior ER visits
er_visits = encounters[encounters['ENCOUNTERCLASS'] == 'emergency'].groupby('PATIENT').size().reset_index(name='PRIOR_ER_VISITS')
inpatient = inpatient.merge(er_visits, on='PATIENT', how='left')
inpatient['PRIOR_ER_VISITS'] = inpatient['PRIOR_ER_VISITS'].fillna(0)

# Prior admissions
inpatient['PRIOR_ADMISSIONS'] = inpatient.groupby('PATIENT').cumcount()

# Is oncology patient
oncology_keywords = ['neoplasm', 'cancer', 'malignant', 'tumor', 'carcinoma', 'lymphoma']
oncology_pattern = '|'.join(oncology_keywords)
oncology_patients = conditions[conditions['DESCRIPTION'].str.lower().str.contains(oncology_pattern, na=False)]['PATIENT'].unique()
inpatient['IS_ONCOLOGY'] = inpatient['PATIENT'].isin(oncology_patients).astype(int)

print("Clinical features added!")
print(inpatient[['TOTAL_CONDITIONS', 'TOTAL_MEDICATIONS', 
                  'TOTAL_PROCEDURES', 'PRIOR_ER_VISITS', 
                  'PRIOR_ADMISSIONS', 'IS_ONCOLOGY']].describe())

Clinical features added!
       TOTAL_CONDITIONS  TOTAL_MEDICATIONS  TOTAL_PROCEDURES  PRIOR_ER_VISITS  \
count       1838.000000        1838.000000       1838.000000      1838.000000   
mean          11.267138          92.134929         69.599565         2.996192   
std            4.121444         108.899315         57.355690         4.827733   
min            2.000000           0.000000          1.000000         0.000000   
25%            9.000000          19.000000         28.000000         1.000000   
50%           10.000000          54.000000         53.000000         2.000000   
75%           13.000000         125.000000        115.000000         4.000000   
max           25.000000        1237.000000        478.000000       105.000000   

       PRIOR_ADMISSIONS  IS_ONCOLOGY  
count       1838.000000  1838.000000  
mean          13.831882     0.652339  
std           13.955467     0.476357  
min            0.000000     0.000000  
25%            2.000000     0.000000  
50%        

In [7]:
# Cell 5 — Build Final Feature Matrix
feature_cols = [
    'AGE', 'GENDER_ENCODED', 'RACE_ENCODED', 'MARITAL_ENCODED',
    'HEALTHCARE_EXPENSES', 'HEALTHCARE_COVERAGE',
    'LENGTH_OF_STAY', 'PRIOR_ADMISSIONS', 'PRIOR_ER_VISITS',
    'TOTAL_CONDITIONS', 'TOTAL_MEDICATIONS', 'TOTAL_PROCEDURES',
    'IS_ONCOLOGY'
]

target_col = 'READMITTED_30'

# Build final dataframe
df_model = inpatient[feature_cols + [target_col]].copy()

# Check for missing values
print("Missing values:")
print(df_model.isnull().sum())
print(f"\nFinal dataset shape: {df_model.shape}")
print(f"\nTarget distribution:")
print(df_model[target_col].value_counts())
print(f"\nClass balance: {df_model[target_col].mean():.2%} positive")

Missing values:
AGE                    0
GENDER_ENCODED         0
RACE_ENCODED           0
MARITAL_ENCODED        0
HEALTHCARE_EXPENSES    0
HEALTHCARE_COVERAGE    0
LENGTH_OF_STAY         0
PRIOR_ADMISSIONS       0
PRIOR_ER_VISITS        0
TOTAL_CONDITIONS       0
TOTAL_MEDICATIONS      0
TOTAL_PROCEDURES       0
IS_ONCOLOGY            0
READMITTED_30          0
dtype: int64

Final dataset shape: (1838, 14)

Target distribution:
READMITTED_30
0    1069
1     769
Name: count, dtype: int64

Class balance: 41.84% positive


In [8]:
# Cell 6 — Save Processed Feature Matrix
df_model.to_csv('../data/processed/features.csv', index=False)
print("Feature matrix saved to data/processed/features.csv")
print(f"Shape: {df_model.shape}")
print("\nFeature columns:")
for col in feature_cols:
    print(f"  - {col}")

Feature matrix saved to data/processed/features.csv
Shape: (1838, 14)

Feature columns:
  - AGE
  - GENDER_ENCODED
  - RACE_ENCODED
  - MARITAL_ENCODED
  - HEALTHCARE_EXPENSES
  - HEALTHCARE_COVERAGE
  - LENGTH_OF_STAY
  - PRIOR_ADMISSIONS
  - PRIOR_ER_VISITS
  - TOTAL_CONDITIONS
  - TOTAL_MEDICATIONS
  - TOTAL_PROCEDURES
  - IS_ONCOLOGY
